# Experiment 2 — Random-Effects Sensitivity Analysis

**Purpose:** Resolve the fixed-effects generalizability gap identified in Experiment 2 (exp2_analysis_v3.ipynb).
In the primary analysis, `model` (LLM identity) is included as a fixed categorical predictor with
6 levels (5 dummy contrasts). This limits population-level inference to the six specific architectures
tested. This notebook re-fits the three primary outcome models (reading hallucination, recognition
accuracy, and metacognitive sensitivity γ) with `model` as a random intercept instead.

**Sensitivity specification:** `model` is removed from fixed effects and replaced with `(1 | model)`,
creating crossed random effects alongside the existing `trace`-level clustering:

| Outcome     | Primary random effects              | Sensitivity random effects              |
|-------------|-------------------------------------|-----------------------------------------|
| RH          | `(1 \| trace)`                       | `(1 \| model) + (1 \| trace)`            |
| Accuracy    | `(1 + source_test \| trace)`         | `(1 \| model) + (1 \| trace)` (fallback) |
| Gamma       | `(1 \| trace)`                       | `(1 \| model) + (1 \| trace)`            |

**Key questions:**
1. Do the primary fixed-effect inferences (source_test, setsize, fb_exp, interactions) replicate
   under the random-model specification?
2. What proportion of outcome variance is attributable to model identity (between-model ICC)?
3. Does treating model as random resolve the generalizability concern raised in the Limitations?

---
## 0. Imports & Setup

In [ ]:
import re
import gc
import warnings
import numpy as np
import pandas as pd
import polars as pl
from IPython.display import display, Markdown
from scipy.stats import norm

import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.packages import importr
from rpy2.robjects.conversion import localconverter

from pymer4.models import lmer, glmer, compare

import rmllm
from rmllm import gamma as gamma_mod

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
np.random.seed(42)

def _r_pkg(name):
    try:
        return importr(name)
    except Exception:
        warnings.warn(f"R package '{name}' not available — diagnostics skipped.")
        return None

_lme4   = importr('lme4')
_base_r = importr('base')
_DHARMa = _r_pkg('DHARMa')

# Optimiser controls
_BOBYQA_G    = "glmerControl(optimizer='bobyqa',    optCtrl=list(maxfun=500000))"
_NLOPTWRAP_G = "glmerControl(optimizer='nloptwrap', optCtrl=list(maxfun=500000))"
_BOBYQA_L    =  "lmerControl(optimizer='bobyqa',    optCtrl=list(maxfun=500000))"
_NLOPTWRAP_L =  "lmerControl(optimizer='nloptwrap', optCtrl=list(maxfun=500000))"

data_dir = rmllm.config.PROCESSED_DATA_DIR
print('Environment ready.')

---
## 1. Data Loading & Preprocessing

Identical preprocessing to `exp2_analysis_v3.ipynb`.

In [ ]:
df = pd.read_csv(data_dir / 'exp2_trial_data.csv')

for col in ['rating', 'confidence', 'order', 'trial_compliance', 'accuracy']:
    df[f'{col}_num'] = pd.to_numeric(df.get(col), errors='coerce')

df['trial_compliance']   = df['trial_compliance_num']
df['accuracy']           = df['accuracy_num']
df['read_hallucination'] = 1 - df['trial_compliance_num']
df['order_c']            = df['order_num'] - 1

_rating_mean  = df['rating_num'].mean()
df['rating_cen'] = df['rating_num'] - _rating_mean

for col in ['setsize', 'fb_exp', 'model', 'source_test']:
    if col in df.columns:
        df[col] = df[col].astype(str)

df_perc = df[df['source_test'] == 'test:perceived'].copy()

SIM_ID = 'trace'
print(f'All trials  : {len(df):,}   Models: {df["model"].nunique()}  '
      f'Traces: {df[SIM_ID].nunique():,}')
print(f'Perceived   : {len(df_perc):,}  (RH rate: {df_perc["read_hallucination"].mean():.3f})')
print(f'Model levels: {sorted(df["model"].unique())}')

---
## 2. Helper Functions

In [ ]:
def _to_polars(data):
    pf = pl.from_pandas(data) if isinstance(data, pd.DataFrame) else data
    cat_cols = ['source_test', 'setsize', 'fb_exp', 'model']
    casts = {c: pl.String for c in cat_cols if c in pf.columns}
    return pf.cast(casts) if casts else pf


def _setup_factors(m):
    cols = m.data.columns if hasattr(m.data, 'columns') else []
    factors = {}
    if 'source_test' in cols:
        factors['source_test'] = ['test:perceived', 'test:imagined']
    if 'setsize'     in cols:
        factors['setsize']     = ['20', '40']
    if 'fb_exp'      in cols:
        factors['fb_exp']      = ['False', 'True']
    if factors:
        m.set_factors(factors)


def is_singular(m):
    try:
        return bool(_lme4.isSingular(m.r_model)[0])
    except Exception:
        return False


def _has_conv_failure(m):
    conv_str = str(getattr(m, 'convergence_status', ''))
    try:
        m_grad = re.search(r'gradient[^\n]*\n\[1\]\s+(\S+)', conv_str, re.DOTALL)
        if m_grad:
            val_str = m_grad.group(1)
            if val_str.upper() != 'NA':
                return float(val_str) > 0.002
    except Exception:
        pass
    return False


def show_fe(m, label='', exponentiate=False):
    tbl = m.result_fit.to_pandas() if isinstance(m.result_fit, pl.DataFrame) else m.result_fit
    p_col = next((c for c in tbl.columns
                  if c.lower() in ('p_value', 'p', 'pr(>|z|)', 'pr(>|t|)')), None)
    if p_col:
        tbl['sig'] = tbl[p_col].map(
            lambda p: '***' if pd.notnull(p) and p < .001
                      else '**'  if pd.notnull(p) and p < .01
                      else '*'   if pd.notnull(p) and p < .05
                      else '')
    if exponentiate:
        for raw, name in [('estimate', 'OR'), ('Estimate', 'OR'),
                          ('conf_low', 'OR_lo'), ('conf_high', 'OR_hi')]:
            if raw in tbl.columns:
                tbl[name] = np.exp(tbl[raw])
    display(Markdown(f'**{label} — Fixed effects**'))
    display(tbl.round(4))
    return tbl


def run_anova(m, label=''):
    try:
        m.anova()
        tbl   = m.result_anova.to_pandas()
        p_col = next((c for c in tbl.columns
                      if c.lower() in ('p_value', 'p', 'p.value', 'pr(>f)', 'pr(>chisq)')), None)
        if p_col:
            tbl['sig'] = tbl[p_col].map(
                lambda p: '***' if pd.notnull(p) and p < .001
                          else '**'  if pd.notnull(p) and p < .01
                          else '*'   if pd.notnull(p) and p < .05
                          else '')
        display(Markdown(f'**{label} — Type-III tests**'))
        display(tbl.round(4))
        return tbl
    except Exception as e:
        print(f'  [ANOVA — {e}]')
        return None


def extract_var_components(m, label=''):
    """Extract variance components from a fitted lme4 model and compute ICCs.

    Note: R identifiers cannot start with underscore; use 'tmpmod', not '_tmp_model'.
    For binomial GLMMs (logit link) the residual variance is not estimated;
    the latent-variable ICC denominator adds pi^2/3 ≈ 3.29.
    """
    display(Markdown(f'**{label} — Variance components (VarCorr)**'))
    r_mod = getattr(m, 'r_model', None)
    if r_mod is None:
        print('  r_model not available.')
        return None
    try:
        ro.globalenv['tmpmod'] = r_mod
        vc_r = ro.r('as.data.frame(VarCorr(tmpmod))')
        with localconverter(ro.default_converter + pandas2ri.converter):
            vc = ro.conversion.rpy2py(vc_r)
        display(vc.round(6))

        model_var = float(vc[vc['grp'] == 'model']['vcov'].values[0]) \
                    if 'model' in vc['grp'].values else 0.0
        trace_var = float(vc[vc['grp'] == 'trace']['vcov'].values[0]) \
                    if 'trace' in vc['grp'].values else 0.0
        has_residual = 'Residual' in vc['grp'].values
        total_random = float(vc['vcov'].sum())

        if has_residual:
            # Gaussian LMM
            total_var = total_random
            icc_model = model_var / total_var if total_var > 0 else np.nan
            icc_trace = trace_var / total_var if total_var > 0 else np.nan
            print(f'  ICC_model (Gaussian) = {icc_model:.4f}  (σ²_model={model_var:.4f})')
            print(f'  ICC_trace (Gaussian) = {icc_trace:.4f}  (σ²_trace={trace_var:.4f})')
            print(f'  Total σ²             = {total_var:.4f}')
        else:
            # Binomial GLMM — latent-variable approach
            pi2_3    = (np.pi ** 2) / 3   # ≈ 3.2899
            total_var = total_random + pi2_3
            icc_model = model_var / total_var if total_var > 0 else np.nan
            icc_trace = trace_var / total_var if total_var > 0 else np.nan
            print(f'  ICC_model (latent) = {icc_model:.4f}  (σ²_model={model_var:.4f})')
            print(f'  ICC_trace (latent) = {icc_trace:.4f}  (σ²_trace={trace_var:.4f})')
            print(f'  σ²_total (latent)  = {total_var:.4f}  [includes π²/3={pi2_3:.4f}]')

        return {'model_var': model_var, 'trace_var': trace_var,
                'total_var': total_var, 'icc_model': icc_model, 'icc_trace': icc_trace}
    except Exception as e:
        print(f'  [VarCorr extraction failed: {e}]')
        return None


def fisher_z(g):
    return np.arctanh(np.clip(g, -0.9999, 0.9999))


print('Helpers defined.')


---
## 3. Model 1 — Reading Hallucination GLMM

**Subset:** perceived-source trials only (N ≈ 36,000)  
**Family:** binomial logit  

**Primary formula (exp2_analysis_v3.ipynb):**
> `read_hallucination ~ setsize * fb_exp + model + order_c + (1 | trace)`

**Sensitivity formula:**
> `read_hallucination ~ setsize * fb_exp + order_c + (1 | model) + (1 | trace)`

Model identity is removed from fixed effects and replaced with a random intercept.
Crossed random effects: trace-level clustering is retained; model-level clustering is added.  
Key inferences: setsize × fb_exp interaction.

In [ ]:
RH_FORMULA  = "read_hallucination ~ setsize * fb_exp + order_c + (1 | model) + (1 | trace)"
rh_cols     = ['read_hallucination', 'setsize', 'fb_exp', 'order_c', 'model', 'trace']
rh_data     = df_perc[rh_cols].dropna()

print(f'Reading Hallucination (perceived only): n = {len(rh_data):,}')
print(f'  RH rate = {rh_data["read_hallucination"].mean():.3f}')
print(f'  Fitting: {RH_FORMULA}')
print(f'  (Expected run time: ~2-5 min for n={len(rh_data):,} with crossed RE)')

m_rh = glmer(RH_FORMULA, data=_to_polars(rh_data), family='binomial')
_setup_factors(m_rh)
m_rh.fit(control=_BOBYQA_G)

if is_singular(m_rh):
    print('  ⚠  isSingular — between-model or between-trace variance on boundary.')
    print('     With n_model=6, singularity of model RE is common; inspect VarCorr.')
if _has_conv_failure(m_rh):
    print('  ⚠  Convergence failure — retrying with nloptwrap...')
    m_rh2 = glmer(RH_FORMULA, data=_to_polars(rh_data), family='binomial')
    _setup_factors(m_rh2)
    m_rh2.fit(control=_NLOPTWRAP_G)
    if not _has_conv_failure(m_rh2):
        m_rh = m_rh2
        print('  ✓  nloptwrap converged.')

try:
    display(m_rh.result_fit_stats.to_pandas().round(4))
except Exception:
    pass

show_fe(m_rh, 'RH — GLMM (1|model)+(1|trace)', exponentiate=True)
vc_rh = extract_var_components(m_rh, 'RH')
run_anova(m_rh, 'RH')

gc.collect()
try:
    ro.r('gc(verbose=FALSE)')
except Exception:
    pass

---
## 4. Model 2 — Recognition Accuracy GLMM

**Subset:** all trials (N = 72,000)  
**Family:** binomial logit  

**Primary formula:**
> `accuracy ~ source_test * setsize + source_test * fb_exp + setsize * fb_exp`  
> `+ model + order_c + rating_cen + (1 + source_test | trace)` (or `(1|trace)` fallback)

**Sensitivity formula:**
> `accuracy ~ source_test * setsize + source_test * fb_exp + setsize * fb_exp`  
> `+ order_c + rating_cen + (1 | model) + (1 | trace)`

**Notes:**
- The maximal trace RE `(1 + source_test | trace)` is dropped here to avoid convergence issues
  with three-level crossed random effects (model + trace slope + trace intercept).
  `(1 | trace)` retains the key trace-level clustering while adding the model-level RE.
- This is the most computationally intensive model (N=72,000 × 3 RE terms).
- Expected run time: 10–30 minutes depending on hardware.

In [ ]:
_ACC_FE_INT = ("source_test + setsize + fb_exp "
               "+ source_test:setsize + source_test:fb_exp + setsize:fb_exp "
               "+ order_c + rating_cen")
ACC_FORMULA = f"accuracy ~ {_ACC_FE_INT} + (1 | model) + (1 | trace)"

acc_cols  = ['accuracy', 'source_test', 'setsize', 'fb_exp',
             'order_c', 'rating_cen', 'model', 'trace']
acc_data  = df[acc_cols].dropna()

print(f'Recognition Accuracy: n = {len(acc_data):,}')
print(f'  Accuracy rate = {acc_data["accuracy"].mean():.3f}')
print(f'  Formula: {ACC_FORMULA}')
print(f'  ⚠  Estimated run time: 10–30 min on typical hardware.')

m_acc = glmer(ACC_FORMULA, data=_to_polars(acc_data), family='binomial')
_setup_factors(m_acc)
m_acc.fit(control=_BOBYQA_G)

if is_singular(m_acc):
    print('  ⚠  isSingular — inspect VarCorr below.')
if _has_conv_failure(m_acc):
    print('  ⚠  Convergence failure — retrying with nloptwrap...')
    m_acc2 = glmer(ACC_FORMULA, data=_to_polars(acc_data), family='binomial')
    _setup_factors(m_acc2)
    m_acc2.fit(control=_NLOPTWRAP_G)
    if not _has_conv_failure(m_acc2):
        m_acc = m_acc2
        print('  ✓  nloptwrap converged.')

try:
    display(m_acc.result_fit_stats.to_pandas().round(4))
except Exception:
    pass

show_fe(m_acc, 'Accuracy — GLMM (1|model)+(1|trace)', exponentiate=True)
vc_acc = extract_var_components(m_acc, 'Accuracy')
run_anova(m_acc, 'Accuracy')

gc.collect()
try:
    ro.r('gc(verbose=FALSE)')
except Exception:
    pass

---
## 5. Model 3 — Metacognitive Sensitivity γ LMM

**Outcome:** Fisher-Z-transformed Goodman–Kruskal γ, computed per trace × source_test  
**Family:** Gaussian identity  

**Primary formula:**
> `f_gamma ~ source_test + setsize + fb_exp + source_test:setsize + source_test:fb_exp`  
> `+ setsize:fb_exp + model + (1 | trace)`

**Sensitivity formula:**
> `f_gamma ~ source_test + setsize + fb_exp + source_test:setsize + source_test:fb_exp`  
> `+ setsize:fb_exp + (1 | model) + (1 | trace)`

**Notes:**
- γ is computed at the trace×source_test level, yielding ~12,000 observations.
- With 2 observations per trace cluster (one per source), a random slope for source_test
  is structurally unidentifiable; `(1|trace)` is the appropriate RE structure.
- The model RE `(1|model)` is crossed with trace RE (each model has many traces).
- Expected run time: <5 minutes for this aggregated outcome.

In [ ]:
# Compute gamma across groups (replicates exp2_analysis_v3 Section 12)
grp_between = ['model', 'setsize', 'fb_exp']
grp_within  = ['source_test']

df_gamma_src = gamma_mod.calculate_gamma_across_groups(
    df, [SIM_ID] + grp_between + grp_within
)
df_gamma_src['f_gamma'] = fisher_z(df_gamma_src['gamma'])

gamma_fit = df_gamma_src.copy()
if 'trace' not in gamma_fit.columns and SIM_ID in gamma_fit.columns:
    gamma_fit = gamma_fit.rename(columns={SIM_ID: 'trace'})
for col in ['setsize', 'fb_exp', 'model', 'source_test']:
    if col in gamma_fit.columns:
        gamma_fit[col] = gamma_fit[col].astype(str)

gamma_data = gamma_fit[['f_gamma', 'source_test', 'setsize', 'fb_exp',
                         'model', 'trace']].dropna()
print(f'Gamma: n = {len(gamma_data):,}  '
      f'(traces: {gamma_data["trace"].nunique():,}, '
      f'models: {gamma_data["model"].nunique()})')

_GAM_FE = ("source_test + setsize + fb_exp "
           "+ source_test:setsize + source_test:fb_exp + setsize:fb_exp")
GAM_FORMULA = f"f_gamma ~ {_GAM_FE} + (1 | model) + (1 | trace)"

print(f'Formula: {GAM_FORMULA}')

m_gam = lmer(GAM_FORMULA, data=_to_polars(gamma_data))
_setup_factors(m_gam)
m_gam.fit(control=_BOBYQA_L)

if is_singular(m_gam):
    print('  ⚠  isSingular — between-model variance may be zero; inspect VarCorr.')
if _has_conv_failure(m_gam):
    print('  ⚠  Convergence failure — retrying with nloptwrap...')
    m_gam2 = lmer(GAM_FORMULA, data=_to_polars(gamma_data))
    _setup_factors(m_gam2)
    m_gam2.fit(control=_NLOPTWRAP_L)
    if not _has_conv_failure(m_gam2):
        m_gam = m_gam2
        print('  ✓  nloptwrap converged.')

try:
    display(m_gam.result_fit_stats.to_pandas().round(4))
except Exception:
    pass

show_fe(m_gam, 'Gamma — LMM (1|model)+(1|trace)')
vc_gam = extract_var_components(m_gam, 'Gamma')
run_anova(m_gam, 'Gamma')

gc.collect()
try:
    ro.r('gc(verbose=FALSE)')
except Exception:
    pass

---
## 6. Variance Component Summary

In [ ]:
summary_rows = []
for label, vc in [('Reading Hallucination', vc_rh),
                  ('Recognition Accuracy',  vc_acc),
                  ('Metacognitive γ',       vc_gam)]:
    if vc is None:
        summary_rows.append({'Outcome': label, 'σ²_model': 'NA',
                             'σ²_trace': 'NA', 'σ²_total': 'NA',
                             'ICC_model': 'NA', 'ICC_trace': 'NA'})
        continue
    summary_rows.append({
        'Outcome':   label,
        'σ²_model':  round(vc.get('model_var',  np.nan), 4),
        'σ²_trace':  round(vc.get('trace_var',  np.nan), 4),
        'σ²_total':  round(vc.get('total_var',  np.nan), 4),
        'ICC_model': round(vc.get('icc_model',  np.nan), 4),
        'ICC_trace': round(vc.get('icc_trace',  np.nan), 4),
    })

icc_tbl = pd.DataFrame(summary_rows)
display(Markdown('**Variance Component Summary — Between-Model and Between-Trace ICCs**'))
display(icc_tbl)

print()
print('Interpretation:')
print('  ICC_model = σ²_model / σ²_total : proportion of variance between LLM architectures')
print('  ICC_trace = σ²_trace / σ²_total : proportion of variance between conversation traces')
print()
print('Note: With n=6 model levels, σ²_model estimates carry substantial uncertainty.')
print('      Treat ICC_model as indicative; confidence intervals require many more model levels.')

---
## 7. Inference Replication Summary

Compare the key fixed-effect inferences from the primary analysis (exp2_analysis_v3.ipynb)
against this random-effects sensitivity analysis. The table below uses the ANOVA results from
Sections 3–5 above.

In [ ]:
display(Markdown('''
### Key Fixed-Effect Inferences — Comparison Table

Fill in the "Random-effects p" and "Replicates?" columns from the ANOVA output above.

| Outcome | Predictor | Primary analysis | Random-effects p | Replicates? |
|---|---|---|---|---|
| RH | setsize | χ²(1)=84.7, p<.001 | see Sec 3 ANOVA | check |
| RH | fb_exp | χ²(1)=96.5, p<.001 | see Sec 3 ANOVA | check |
| RH | setsize×fb_exp | χ²(1)=17.3, p<.001 | see Sec 3 ANOVA | check |
| Accuracy | source_test | χ²(1)=962, p<.001 | see Sec 4 ANOVA | check |
| Accuracy | setsize | χ²(1)=84.7, p<.001 | see Sec 4 ANOVA | check |
| Accuracy | fb_exp | χ²(1)=96.5, p<.001 | see Sec 4 ANOVA | check |
| Accuracy | source_test×fb_exp | χ²(1)=857, p<.001 | see Sec 4 ANOVA | check |
| Accuracy | setsize×fb_exp | χ²(1)=17.3, p<.001 | see Sec 4 ANOVA | check |
| Gamma | source_test | F=?, p<.001 | see Sec 5 ANOVA | check |
| Gamma | fb_exp | F=?, p<.001 | see Sec 5 ANOVA | check |
| Gamma | source_test×fb_exp | F=?, p<.001 | see Sec 5 ANOVA | check |

*Update after running the notebook.*
'''))

# Report AIC of sensitivity models
print('AIC of random-effects models:')
for label, m in [('RH (random)', m_rh), ('Accuracy (random)', m_acc),
                 ('Gamma (random)',    m_gam)]:
    try:
        aic = float(m.result_fit_stats['AIC'][0])
        print(f'  {label}: AIC = {aic:.1f}')
    except Exception:
        print(f'  {label}: AIC unavailable')

---
## 8. Conclusions

### Interpretation framework

**Scenario A — ICC_model is low (< 0.05) and all inferences replicate:**  
LLM architecture accounts for a small fraction of outcome variance. Fixed-effect inferences
about source monitoring, feedback, and set size are robust to model specification. The
generalizability limitation is primarily conceptual: conclusions apply to these six architectures
in a narrow statistical sense, but the empirical pattern is stable across model-as-random
specification. **Manuscript update:** revise Limitations to note that sensitivity analysis
confirms robustness; acknowledge imprecision of between-model variance estimate.

**Scenario B — ICC_model is moderate (0.05–0.20) and inferences replicate:**  
Between-model variance is non-trivial but does not change the fixed-effect conclusions.
The random-effects model provides a better characterization of the data structure, and
population-level inference is justified (with appropriate caveats about n=6 model levels).
**Manuscript update:** report both fixed- and random-effects results; note that
model identity explained X% of variance.

**Scenario C — ICC_model is high (> 0.20) or inferences do not replicate:**  
Model identity is a dominant source of variance. Fixed-effect conclusions about experimental
manipulations may be confounded with LLM-architecture differences. Stronger generalizability
claims require testing a larger and more diverse set of models. **Manuscript update:** revise
Results to caveat which effects replicate and which do not; expand Limitations accordingly.

**Singularity note:**  
If `(1|model)` is singular (σ²_model ≈ 0), the between-model variance is indistinguishable
from zero given n=6 levels. This can mean: (a) models are genuinely homogeneous on this
outcome (after accounting for fixed effects), or (b) n=6 is insufficient to estimate σ²_model
reliably. In either case, the fixed-effects model is a reasonable approximation. Report the
attempt and the singularity; do not overinterpret the null variance estimate.

### Recommendation for manuscript

Run this notebook to completion, then update `main.tex` as follows:

1. **Methods — Data Analysis:** Add one paragraph describing the random-effects sensitivity
   analysis (what was changed, why, what was compared).
2. **Results — corresponding sections:** Add a brief note after each primary result confirming
   whether the fixed-effect inference replicated under the random-model specification and
   reporting the between-model ICC.
3. **Limitations:** Replace or supplement the fixed-effects limitation note with a summary
   of what the sensitivity analysis found, and note whether any inferences changed.